In [1]:
import pandas as pd
df = pd.read_json("../data/results/codetrans_m1_results.jsonl", lines=True)

In [2]:
import re

def extract_code(text: str) -> str:
    # Pattern explanation:
    # ```(?:\w+)? -> Matches opening backticks and optional language name (non-capturing)
    # \s* -> Matches optional whitespace/newline after the language tag
    # (.*?)        -> Captures the actual code (non-greedy)
    # \s*```       -> Matches optional whitespace and closing backticks
    pattern = r"```(?:\w+)?\s*(.*?)\s*```"
    
    # re.DOTALL allows the '.' to match newline characters
    match = re.search(pattern, text, re.DOTALL)
    
    if match:
        return match.group(1).strip()
    
    # If no code block is found, return the original text
    return text

In [3]:
from pygments.lexers import get_lexer_by_name
from pygments.token import Token

def compare_generic_tokens(code1: str, code2: str, lang:  str) -> bool:
    lexer = get_lexer_by_name(lang)

    def extract_logic(code):
        # Filter out Whitespace and Comments
        return [
            tok_value.strip() 
            for tok_type, tok_value in lexer.get_tokens(code)
            if tok_type not in Token.Text and tok_type not in Token.Comment
            and tok_value.strip() != ""
        ]

    return extract_logic(code1) == extract_logic(code2)

In [4]:
df['model_output'] = df['model_output'].apply(lambda x: extract_code(x))

In [5]:
from codebleu import calc_codebleu
import sacrebleu

def bleu_score(pred, ref):
  return sacrebleu.sentence_bleu(pred, [ref]).score

def codebleu_score(pred, ref, lang):
    res = calc_codebleu(
        [ref],
        [pred],
        lang
    )
    return res["codebleu"]

In [6]:
from rouge_score import rouge_scorer, scoring
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l_aggregated(preds, refs):
  aggregator = scoring.BootstrapAggregator()
  for ref, pred in zip(refs, preds):
    aggregator.add_scores(scorer.score(ref, pred))
  return aggregator.aggregate()["rougeL"].mid.fmeasure

In [7]:
df["match"] = df.apply(lambda row: compare_generic_tokens(row['cs_code'], row['model_output'], lang="c#"), axis=1)
df["bleu"] = df.apply(lambda x: bleu_score(x.model_output, x.cs_code) / 100.0, axis=1)
df["codebleu"] = df.apply(lambda x: codebleu_score(x.model_output, x.cs_code, lang="c_sharp"), axis=1)
df["rougeL"] = [rouge_l_aggregated([p], [r]) for p, r in zip(df.model_output, df.cs_code)]

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1400 entries, 0 to 1399
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            1400 non-null   int64  
 1   java_code     1400 non-null   object 
 2   cs_code       1400 non-null   object 
 3   model_name    1400 non-null   object 
 4   prompt_type   1400 non-null   object 
 5   model_output  1400 non-null   object 
 6   match         1400 non-null   bool   
 7   bleu          1400 non-null   float64
 8   codebleu      1400 non-null   float64
 9   rougeL        1400 non-null   float64
dtypes: bool(1), float64(3), int64(1), object(5)
memory usage: 99.9+ KB


In [9]:
import pandas as pd
from scipy import stats as scipy_stats

# 1. Clean data and set metrics
df['model_name'] = df['model_name'].str.split('/').str[-1]
metrics = ['match', 'bleu', 'codebleu', 'rougeL']

# 2. Calculate Mean and Std
agg_stats = df.groupby(['model_name', 'prompt_type'])[metrics].agg(['mean', 'std'])

# 3. Format the strings (mean ± std)
display_df = pd.DataFrame(index=agg_stats.index)
for m in metrics:
    display_df[m] = agg_stats.apply(lambda x: f"{x[m]['mean']:.3f} ± {x[m]['std']:.3f}", axis=1)

# 4. Significance Styling Logic
def style_significance(data):
    style_df = pd.DataFrame('', index=data.index, columns=data.columns)
    
    # Iterate through unique models in the MultiIndex level 0
    for model in data.index.get_level_values(0).unique():
        model_mask = df['model_name'] == model
        group1_raw = df[model_mask & (df['prompt_type'] == 'Few-shot')]
        group2_raw = df[model_mask & (df['prompt_type'] == 'Zero-shot')]
        
        for m in metrics:
            if not group1_raw.empty and not group2_raw.empty:
                # Perform T-Test
                _, p_val = scipy_stats.ttest_ind(group1_raw[m], group2_raw[m], nan_policy='omit')
                
                if p_val < 0.05:
                    # Find which prompt had the higher mean
                    m1, m2 = group1_raw[m].mean(), group2_raw[m].mean()
                    better_prompt = 'Few-shot' if m1 > m2 else 'Zero-shot'
                    style_df.loc[(model, better_prompt), m] = 'font-weight: bold'
    return style_df

# 5. Display with MultiIndex (this handles the cell merging)
styled_table = display_df.style.apply(style_significance, axis=None)

# Optional: Add borders to make the merging look even cleaner
styled_table.set_table_styles([
    {'selector': 'th', 'props': [('border', '1px solid lightgrey')]},
    {'selector': 'td', 'props': [('border', '1px solid lightgrey')]}
])

styled_table

# AI Judge

In [10]:
prompt_judge = """
You are a judge tasked with detecting potential cheating among students.

Task Description:
The assignment is to translate {src_lang} code into {tgt_lang}. You will be given:

1. One {src_lang} code snippet (the original source)
2. A {tgt_lang} translation from Student A
3. A {tgt_lang} translation from Student B

Your job is to determine whether Student B copied ideas from Student A.

Focus ONLY on elements in Student B’s solution that could reasonably have been
inferred from Student A’s answer but could NOT have been inferred from:
- the original {src_lang} code, or
- standard {src_lang}-to-{tgt_lang} translation knowledge.

Do NOT penalize common patterns, idiomatic translations, or stylistic similarities
that are typical for {src_lang}-to-{tgt_lang} conversions.

Evaluation Rubric:
(Score each category from 0–2, and explain the reasoning for that score.)

1. Derivability from {src_lang}
   0 → Fully inferable from {src_lang}
   1 → Slightly non-obvious but plausible
   2 → Not inferable from {src_lang}

2. Non-Obvious Design Choices
   0 → Standard / idiomatic
   1 → Questionable but explainable
   2 → Distinctive and unnecessary, matches Student A

3. Shared Errors or Quirks
   0 → None shared
   1 → One shared oddity
   2 → Multiple shared mistakes or quirks

4. Optional or Advanced Features
   0 → Expected usage
   1 → Similar optional usage
   2 → Same unnecessary advanced features

Scoring Interpretation:
0–3   → No evidence of copying
4–6   → Suspicious similarities
7–8   → Strong evidence Student B copied ideas from Student A

Final Rule:
Conclude copying ONLY if similarities are unlikely to have arisen independently.

You MUST:
- Assign a score (0–2) for each rubric category
- Provide a brief explanation justifying each score
- Compute the total score
- State the final conclusion based on the scoring interpretation

Original source {src_lang}:
{src}

{tgt_lang} translation from Student A:
{target}

{tgt_lang} translation from Student B:
{output}

You must answer with the following JSON format. Write only the JSON output in your response
{{
  "rubric": {{
    "derivability_from_source": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }},
    "non_obvious_design_choices": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }},
    "shared_errors_or_quirks": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }},
    "optional_or_advanced_features": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }}
  }},
  "total_score": <sum of all scores>,
}}
"""

In [24]:
from litellm import completion
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def process_row(idx, row):
    try:
        formatted_prompt = prompt_judge.format(
            src=row['java_code'],
            target=row['cs_code'],
            output=row['model_output'],
            src_lang='Java',
            tgt_lang='C#'
        )

        response = completion(
            # model="openrouter/openai/gpt-oss-120b",
            model="openrouter/openai/gpt-5-mini",
            messages=[{"role": "user", "content": formatted_prompt}],
            num_retries=5
        )

        return idx, response.choices[0].message.content

    except Exception as e:
        return idx, f"ERROR: {str(e)}"


df['judge_gpt_5_mini'] = None

MAX_WORKERS = 10  # adjust based on rate limits

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(process_row, idx, row)
        for idx, row in df.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx, result = future.result()
        df.at[idx, 'judge_gpt_5_mini'] = result

print(f"✓ Complete! Processed all {len(df)} rows")


100%|██████████| 1400/1400 [44:46<00:00,  1.92s/it] 

✓ Complete! Processed all 1400 rows


In [36]:
df.to_json("m1_metrics.jsonl", orient="records", lines=True)

In [14]:
import re
import json
def extract_json(text: str):
    """
    Extracts the first valid JSON object from a string.
    Returns a Python dict or list.
    Raises ValueError if no valid JSON is found.
    """

    # 1. Try to extract from fenced code block ```json ... ```
    code_block = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if code_block:
        try:
            return json.loads(code_block.group(1))
        except json.JSONDecodeError:
            pass

    # 2. Fallback: find first {...} or [...]
    stack = []
    start = None

    for i, ch in enumerate(text):
        if ch in "{[":
            if not stack:
                start = i
            stack.append(ch)
        elif ch in "}]":
            if stack:
                stack.pop()
                if not stack and start is not None:
                    candidate = text[start:i + 1]
                    try:
                        return json.loads(candidate)
                    except json.JSONDecodeError:
                        start = None
    return None

In [28]:
df['rubric_gpt_oss'] = df['judge_gpt_oss'].apply(lambda x:extract_json(x))
df['rubric_gpt_5_mini'] = df['judge_gpt_5_mini'].apply(lambda x:extract_json(x))

In [30]:
df['total_score_gpt_oss'] = df['rubric_gpt_oss'].apply(lambda x:x.get('total_score',-1) if x else 0)
df['total_score_gpt_5_mini'] = df['rubric_gpt_5_mini'].apply(lambda x:x.get('total_score',-1) if x else 0)

In [31]:
df['total_score_gpt_oss'].value_counts()

total_score_gpt_oss
 0    1270
 1      72
 2      37
 3      15
 4       4
 6       1
-1       1
Name: count, dtype: int64

In [32]:
df['total_score_gpt_5_mini'].value_counts()

total_score_gpt_5_mini
0    1375
1      21
2       3
3       1
Name: count, dtype: int64